# 2-hour VQE tutorial for intermediate Qiskit users

This notebook combines the IBM Quantum Learning **VQE module** with the **ground-state workflow lesson** into a single, practice-oriented tutorial.

**Audience.** Users who already know basic circuit construction, observables, transpilation, and primitive-based execution in Qiskit.

**Format (rough pacing).**
- 0:00-0:15 — Variational principle and VQE loop
- 0:15-0:45 — One-qubit warm-up: effective H₂ Hamiltonian
- 0:45-1:20 — Four-qubit H₂ example with a hardware-efficient ansatz
- 1:20-1:40 — Hardware-aware transpilation and Runtime pattern
- 1:40-2:00 — Exercises and extensions

**Main learning goals.**
1. Translate a chemistry-motivated Hamiltonian into a Qiskit `SparsePauliOp`.
2. Implement a VQE cost function with Qiskit primitives.
3. Compare exact diagonalization, statevector VQE, and hardware-oriented execution flow.
4. Reason about ansatz choice, optimizer behavior, and transpilation effects.

**Primary source material.**
- IBM Quantum Learning module: *Variational Quantum Eigensolver*
- IBM Quantum Learning course lesson: *Ground state energies / Bringing it all together with Qiskit Runtime*


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.optimize import minimize

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import efficient_su2
from qiskit.primitives import StatevectorEstimator

np.set_printoptions(precision=6, suppress=True)
RNG = np.random.default_rng(7)


## 1. Conceptual refresher: how VQE works

VQE is built on the **variational principle**: for any normalized trial state $|\psi(\theta)\rangle$,

$$
E(\theta) = \langle \psi(\theta)| H |\psi(\theta)\rangle \ge E_0,
$$

where $E_0$ is the true ground-state energy of the Hamiltonian $H$.

That gives a hybrid loop:

1. Choose an ansatz circuit $U(\theta)$.
2. Prepare $|\psi(\theta)\rangle = U(\theta)|0\cdots0\rangle$.
3. Estimate $\langle H \rangle_\theta$ with quantum primitives.
4. Use a classical optimizer to update $\theta$.
5. Repeat until convergence.

The IBM learning materials organize this inside the **Qiskit patterns** workflow:
1. map classical inputs to a quantum problem,
2. optimize for target hardware,
3. execute with primitives,
4. post-process classically.


## 2. Warm-up problem: a one-qubit effective H₂ Hamiltonian

The IBM VQE module uses a reduced one-qubit Hamiltonian for H₂ at bond distance $0.735\ \AA$:

$$
H = -1.04886087\,I -0.7967368\,Z + 0.18121804\,X.
$$

This is a perfect warm-up because:
- the exact answer is easy to compute,
- the ansatz can be visualized clearly,
- the VQE loop is the same as in larger problems.


In [ ]:
h2_1q = SparsePauliOp.from_list(
    [("I", -1.04886087), ("Z", -0.7967368), ("X", 0.18121804)]
)

matrix_1q = h2_1q.to_matrix()
evals_1q, evecs_1q = np.linalg.eigh(matrix_1q)

print("1-qubit H2 Hamiltonian:")
print(matrix_1q)
print("\nEigenvalues:", evals_1q)
print("Exact ground-state energy:", evals_1q[0])


### A compact ansatz

For a one-qubit problem, an expressive but still interpretable ansatz is

$$
|\psi(\theta)\rangle = R_y(\theta_0) R_z(\theta_1) |0\rangle.
$$

Because the Hamiltonian contains only $I, X, Z$, one could even get away with fewer parameters, but keeping two parameters is useful pedagogically.


In [ ]:
theta = ParameterVector("θ", 2)

ansatz_1q = QuantumCircuit(1)
ansatz_1q.ry(theta[0], 0)
ansatz_1q.rz(theta[1], 0)

ansatz_1q.draw("mpl")


### Primitive-based cost function

The IBM lessons use the Estimator primitive through the V2 pub format. We will keep the same structure here so that the notebook scales naturally from local simulation to Runtime execution.


In [ ]:
estimator = StatevectorEstimator()

def energy_from_parameters(params, ansatz, hamiltonian, estimator):
    pub = (ansatz, [hamiltonian], [params])
    result = estimator.run(pubs=[pub]).result()
    return float(np.real(result[0].data.evs[0]))


In [ ]:
x0 = np.array([0.3, 0.1])

history_1q = []

def objective_1q(params):
    energy = energy_from_parameters(params, ansatz_1q, h2_1q, estimator)
    history_1q.append(energy)
    return energy

res_1q = minimize(objective_1q, x0=x0, method="COBYLA")
print(res_1q)
print("\nVQE energy:", res_1q.fun)
print("Exact energy:", evals_1q[0])
print("Absolute error:", abs(res_1q.fun - evals_1q[0]))


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history_1q, marker="o")
plt.xlabel("Iteration")
plt.ylabel("Energy")
plt.title("One-qubit VQE convergence")
plt.grid(True, alpha=0.3)
plt.show()


### Exercise 2
Change the optimizer from `COBYLA` to one of:
- `Nelder-Mead`
- `Powell`
- `BFGS`

Then compare:
1. number of function evaluations,
2. final energy,
3. sensitivity to the initial point.

Record your observations below.


**Your observations:**


## 3. Exact landscape scan for intuition

Since the one-qubit example is tiny, we can visualize the energy landscape directly. This is useful for developing optimizer intuition before moving to a larger ansatz.


In [ ]:
grid = np.linspace(-np.pi, np.pi, 121)
landscape = np.zeros((len(grid), len(grid)))

for i, a in enumerate(grid):
    for j, b in enumerate(grid):
        landscape[i, j] = energy_from_parameters([a, b], ansatz_1q, h2_1q, estimator)

plt.figure(figsize=(6, 5))
plt.imshow(
    landscape,
    extent=[grid[0], grid[-1], grid[0], grid[-1]],
    origin="lower",
    aspect="auto",
)
plt.colorbar(label="Energy")
plt.scatter([res_1q.x[1]], [res_1q.x[0]], marker="x", s=100)
plt.xlabel(r"$\theta_1$")
plt.ylabel(r"$\theta_0$")
plt.title("One-qubit VQE energy landscape")
plt.show()


### Exercise 3
Why is this visualization much easier for the one-qubit example than for a hardware-efficient ansatz on several qubits?

Write a short answer before proceeding.


**Your answer:**


## 4. Four-qubit H₂ example: from the ground-state workflow lesson

The ground-state lesson provides a compact four-qubit Pauli Hamiltonian for H₂, along with the nuclear-repulsion contribution

$$
E_\mathrm{nuc} = 0.7199689944489797.
$$

This is a more realistic VQE workflow:
- the operator is no longer trivial,
- the ansatz has several parameters,
- exact diagonalization is still possible classically for benchmarking.


In [ ]:
H4 = SparsePauliOp(
    [
        "IIII",
        "IIIZ",
        "IZII",
        "IIZI",
        "ZIII",
        "IZIZ",
        "IIZZ",
        "ZIIZ",
        "IZZI",
        "ZZII",
        "ZIZI",
        "YYYY",
        "XXYY",
        "YYXX",
        "XXXX",
    ],
    coeffs=[
        -0.09820182 + 0.0j,
        -0.17407510 + 0.0j,
        -0.17407510 + 0.0j,
         0.22429330 + 0.0j,
         0.22429330 + 0.0j,
         0.16891402 + 0.0j,
         0.12100990 + 0.0j,
         0.16631441 + 0.0j,
         0.16631441 + 0.0j,
         0.12100990 + 0.0j,
         0.17504456 + 0.0j,
         0.04530451 + 0.0j,
         0.04530451 + 0.0j,
         0.04530451 + 0.0j,
         0.04530451 + 0.0j,
    ],
)

nuclear_repulsion = 0.7199689944489797

eigvals_4q = np.linalg.eigvalsh(H4.to_matrix())
electronic_exact = float(np.min(eigvals_4q))
total_exact = electronic_exact + nuclear_repulsion

print("Exact electronic ground-state energy:", electronic_exact)
print("Exact total energy:", total_exact)


### Ansatz selection

The IBM lesson uses `efficient_su2` with linear entanglement and `rx` rotations. That is a good intermediate example because it is easy to modify while still resembling hardware-efficient practice.


In [ ]:
ansatz_4q = efficient_su2(
    num_qubits=H4.num_qubits,
    su2_gates=["rx"],
    entanglement="linear",
    reps=1,
)

print("Number of parameters:", ansatz_4q.num_parameters)
print("Decomposed depth:", ansatz_4q.decompose().depth())
ansatz_4q.decompose().draw("mpl")


In [ ]:
history_4q = []

def objective_4q(params):
    energy = energy_from_parameters(params, ansatz_4q, H4, estimator)
    history_4q.append(energy)
    return energy

x0_4q = 2 * np.pi * RNG.random(ansatz_4q.num_parameters)
res_4q = minimize(
    objective_4q,
    x0=x0_4q,
    method="COBYLA",
    options={"maxiter": 200},
)

electronic_vqe = float(res_4q.fun)
total_vqe = electronic_vqe + nuclear_repulsion

print(res_4q.message)
print("Success:", res_4q.success)
print("Electronic VQE energy:", electronic_vqe)
print("Total VQE energy:", total_vqe)
print("Electronic exact energy:", electronic_exact)
print("Absolute error:", abs(electronic_vqe - electronic_exact))


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history_4q)
plt.axhline(electronic_exact, linestyle="--")
plt.xlabel("Iteration")
plt.ylabel("Electronic energy")
plt.title("Four-qubit VQE convergence")
plt.grid(True, alpha=0.3)
plt.show()


## 5. Post-processing: compare exact vs VQE

The IBM module emphasizes a clean classical post-processing step after primitive execution. Here we summarize the result.


In [ ]:
summary = {
    "electronic_exact": electronic_exact,
    "electronic_vqe": electronic_vqe,
    "electronic_abs_error": abs(electronic_vqe - electronic_exact),
    "total_exact": total_exact,
    "total_vqe": total_vqe,
    "total_abs_error": abs(total_vqe - total_exact),
}

summary


In [ ]:
labels = ["electronic", "total"]
exact_vals = [electronic_exact, total_exact]
vqe_vals = [electronic_vqe, total_vqe]

x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(6, 4))
plt.bar(x - width/2, exact_vals, width, label="Exact")
plt.bar(x + width/2, vqe_vals, width, label="VQE")
plt.xticks(x, labels)
plt.ylabel("Energy")
plt.title("Exact vs VQE energies")
plt.legend()
plt.show()


### Exercise 4
Try at least two of the following modifications and record the effect on convergence and final error.

1. Increase `reps` from `1` to `2`.
2. Replace `su2_gates=["rx"]` with `["ry", "rz"]`.
3. Start from all-zero parameters instead of random ones.
4. Limit `maxiter` to `30`.

Which change helped the most, and why do you think that happened?


**Your observations:**


## 6. Why ansatz design matters

The VQE module explicitly notes that ansatz quality matters because the optimizer can only search inside the family of states your circuit can prepare.

A useful distinction is:
- **chemistry-inspired ansätze**: often more physically meaningful, but deeper,
- **hardware-efficient ansätze**: often shallower, but can be harder to optimize or less chemically structured.

This tradeoff is central to practical VQE.


### Exercise 5
Suppose an ansatz is very expressive but extremely deep on your target backend. Give two concrete reasons why that can still hurt VQE performance on real hardware.


**Your answer:**


## 7. Hardware-aware workflow: transpilation and layout

The ground-state lesson emphasizes that VQE is not just about the abstract circuit. On real hardware, you also need:
- a target backend,
- a transpilation strategy,
- a qubit layout,
- a Hamiltonian transformed to match that layout.

The cell below is **optional** and requires IBM Quantum credentials.


In [ ]:
# Optional hardware section.
# Requires valid IBM Quantum credentials.

# from qiskit_ibm_runtime import QiskitRuntimeService
# from qiskit.transpiler import PassManager
# from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
# from qiskit.transpiler.passes import ALAPScheduleAnalysis, PadDynamicalDecoupling, ConstrainedReschedule
# from qiskit.circuit.library import XGate
#
# service = QiskitRuntimeService(channel="ibm_quantum_platform")
# backend = service.least_busy(operational=True, simulator=False)
# print("Selected backend:", backend.name)
#
# target = backend.target
# pm = generate_preset_pass_manager(target=target, optimization_level=3)
# pm.scheduling = PassManager(
#     [
#         ALAPScheduleAnalysis(target=target),
#         ConstrainedReschedule(
#             acquire_alignment=target.acquire_alignment,
#             pulse_alignment=target.pulse_alignment,
#             target=target,
#         ),
#         PadDynamicalDecoupling(
#             target=target,
#             dd_sequence=[XGate(), XGate()],
#             pulse_alignment=target.pulse_alignment,
#         ),
#     ]
# )
#
# ansatz_isa = pm.run(ansatz_4q)
# H4_isa = H4.apply_layout(ansatz_isa.layout)
#
# ansatz_isa.draw(output="mpl", idle_wires=False, style="iqp")


### Why do we apply the layout to the Hamiltonian?

After transpilation, the logical qubits in your ansatz may be mapped to different physical qubits. If the observable is left in the original logical ordering, the primitive would estimate the wrong operator on the transpiled circuit. Applying the layout keeps the circuit and observable consistent.


### Exercise 6
Explain, in your own words, the difference between:
1. optimizing the **abstract variational parameters** of the ansatz, and
2. optimizing the **compiled implementation** of that ansatz for a backend.


**Your answer:**


## 8. Runtime execution pattern (optional)

The one thing local statevector VQE does **not** teach is how iterative execution is organized efficiently on real hardware. The IBM lessons emphasize using Runtime primitives and session-like execution modes so repeated optimizer steps do not each pay full queuing overhead.

The next cell shows the pattern, but it is intentionally left commented because it will submit remote jobs.


In [ ]:
# Optional Runtime pattern.
#
# from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator
#
# runtime_history = []
#
# def runtime_objective(params, ansatz, hamiltonian, estimator):
#     pub = (ansatz, [hamiltonian], [params])
#     result = estimator.run(pubs=[pub]).result()
#     energy = float(np.real(result[0].data.evs[0]))
#     runtime_history.append(energy)
#     return energy
#
# with Session(backend=backend) as session:
#     estimator_rt = Estimator(mode=session)
#     # estimator_rt.options.default_shots = 2000
#     # estimator_rt.options.resilience_level = 0
#     res_runtime = minimize(
#         runtime_objective,
#         x0_4q,
#         args=(ansatz_isa, H4_isa, estimator_rt),
#         method="COBYLA",
#         options={"maxiter": 30},
#     )
#
# print(res_runtime)


## 9. Suggested discussion prompts for a live tutorial

These work well as short pauses during a 2-hour session.

1. Why does exact diagonalization stop being viable long before primitive execution does?
2. What kinds of Hamiltonian structure suggest a chemistry-inspired ansatz instead of a generic hardware-efficient one?
3. When does a lower-energy local simulator result fail to predict a better hardware result?
4. How do shot noise and optimizer noise interact near convergence?


## 10. Challenge exercises

### Exercise 7 — compute the nuclear-repulsion contribution
The learning module derives the nuclear-repulsion energy for H₂ at bond distance $0.735\ \AA$ as approximately `0.71997` Hartree.

Reproduce that value from
$$
E_\mathrm{nuc} = \frac{1}{R}
$$
when $R$ is expressed in Bohr radii and
$$
1\ \AA = 1.88973\ a_0.
$$

Write a short code cell below.


### Exercise 8 — optimizer overhead by ansatz complexity
Construct a small experiment:
1. `reps = 1`
2. `reps = 2`
3. `reps = 3`

For each, record:
- number of parameters,
- circuit depth after decomposition,
- final energy after a fixed optimization budget, for example `maxiter = 80`.

Then comment on the tradeoff between expressibility and optimization overhead.


### Exercise 9 — noisy estimation
Replace the statevector estimator with a noisy simulation workflow if you have `qiskit-aer` installed.

Suggested directions:
- compare noiseless and noisy energies at the optimized parameter point,
- rerun the optimizer with noise present,
- inspect whether the convergence trace becomes less smooth.


## 11. Optional solution sketches

Use these only after attempting the exercises.

### Exercise 1 sketch
The variational principle states that the expectation value of the Hamiltonian over any normalized trial state is an upper bound on the true ground-state energy. Therefore, ideal VQE can only approach the ground state from above.

### Exercise 3 sketch
The landscape dimension grows with the number of parameters. A two-parameter ansatz can be plotted directly, but an eight-parameter or larger ansatz cannot be visualized as a full landscape without slicing.

### Exercise 5 sketch
A very deep ansatz can accumulate gate error and decoherence, and it can also become harder for the classical optimizer to train because of flatter or more irregular landscapes.

### Exercise 6 sketch
Variational optimization changes the parameter values inside a chosen circuit family. Compilation optimization changes how that circuit is implemented on a specific backend, for example qubit mapping, native-gate synthesis, and scheduling.


## 12. Takeaways

By the end of this notebook you should be able to:
- implement a primitive-based VQE loop,
- benchmark VQE against exact diagonalization on small instances,
- explain why ansatz choice and transpilation both matter,
- separate the **algorithmic** part of VQE from the **hardware-execution** part.

For a next session, a good follow-up is to extend this notebook in one of three directions:
1. bond-length sweeps for H₂,
2. UCC-style chemistry ansätze,
3. Runtime-based execution with shot noise and error-mitigation settings.
